# 1. Read and inspect a trajectory
**Input:** your LAMMPS text dump, explicit units/timestep, optional species/mass data file.
**Output:** a synchronized LammpsFrame and ASE Atoms, plus a small scalar history.
**Modules:** io.lammps -> structure.graph / statistics.md. No saved analysis reports are needed.

Change the configuration cell, then Run All. Install the package with `pip install -e ".[notebooks]"` in your Python environment and select that notebook kernel. Restart the kernel after package changes. Paths may be absolute; Python raw strings or forward slashes avoid Windows escape sequences.

Set `RUN_DIR` to your simulation folder. Leave `INPUT_FILE`, `DUMP_FILE`, and `DATA_FILE` as `None` for automatic discovery. Files directly in the folder take priority; subfolders are searched if necessary. If several files match, set the corresponding filename override. Keep units and other physical settings explicit.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from MartiniFuncs.io.lammps import read_lammps_frame, iter_lammps_frames
from MartiniFuncs.trajectory.analysis import resolve_run_files

RUN_DIR = Path(r"/path/to/simulation")
INPUT_FILE = None  # optional filename override; None discovers automatically
DUMP_FILE = None  # relative to RUN_DIR; set when several dumps exist
DATA_FILE = None  # optional matching data file for masses/elements
UNITS = "metal"
TIMESTEP_PS = 0.0005  # ps per MD step, NOT per saved frame
TYPE_SYMBOLS = None  # optional {1: "C", 2: "H", ...}
FRAME_INDEX = 0
OUTPUT_DIR = RUN_DIR / "analysis"
WRITE_FILES = False
OVERWRITE = False

MAX_FRAMES = 10  # bounded inspection; None streams the entire dump
TEMPERATURE_DOF = None  # e.g. 4077 for the unconstrained 1360-atom example

## Read one frame
The parser sorts by LAMMPS atom ID and aligns positions, forces, velocities, and metadata. `global_index` is persistent (normally LAMMPS ID minus one); an ASE row index changes after slicing/reordering. Coordinates and forces are converted to ASE units. Raw `frame.columns` remain in source units.

### Mathematics: frames, time, units and molecular identity
Let $f$ be the zero-based source frame, $s_f$ its LAMMPS timestep, and $h=$ `TIMESTEP_PS` (ps per MD step). The displayed time is
$$t_f=s_fh.$$
`FRAME_INDEX` / `FRAME_INDICES` select $f$, not $s_f$. Where present, `FRAME_STRIDE` $q$ retains $f\bmod q=0$; `MAX_FRAMES` $F$ reads only $0\le f<F$. `STARTFRAME`, `ENDFRAME` or each `WINDOWS` entry $(a,b)$ keeps retained frames with $a\le f<b$; `None` means the available end. The lag spacing is $\Delta t=t_{j+1}-t_j$ between **retained** samples, not $h$. Paths and write/display switches do not enter these equations.

`UNITS` controls conversion on reading. In these workflows coordinates are Å, masses are atomic mass units, source forces are eV/Å, and reported velocities are Å/ps. `TYPE_SYMBOLS` sets the type-to-element mapping used to infer atomic properties when needed; an incorrect mapping can change masses and bond radii.

Where `BOND_SCALE` $s$ is used (default 1.3), atoms are connected when their periodic-image separation satisfies
$$\|\mathbf r_j+\mathbf nH-\mathbf r_i\|\le s(r_i^{\rm cov}+r_j^{\rm cov}).$$
$H$ contains cell vectors as rows, $\mathbf n$ is an integer image shift, and $r^{\rm cov}$ is the element's covalent radius (Å). Molecules are connected components of this inferred graph; increasing $s$ can merge close molecules. Persistent atom IDs are labels, not coordinate-array row indices.


In [ ]:
files = resolve_run_files(RUN_DIR, input_file=INPUT_FILE, dump_file=DUMP_FILE, data_file=DATA_FILE, require_input=False)
print("Selected files:", files)
frame = read_lammps_frame(files["dump"], index=FRAME_INDEX, units=UNITS,
                          input_datafile=files["data"], type_symbols=TYPE_SYMBOLS)
atoms = frame.atoms
print("Source:", files["dump"])
print(f"Frame {frame.index}; step {frame.timestep}; time {frame.timestep*TIMESTEP_PS:g} ps")

print("Cell (angstrom):", atoms.cell.array)
print("Periodic axes:", atoms.pbc)
print("Origin:", atoms.get_celldisp())
print("Arrays:", sorted(atoms.arrays))
display(pd.DataFrame({"row": np.arange(len(atoms)),
                      "global_index": atoms.arrays["global_index"],
                      "lammps_id": atoms.arrays["lammps_id"],
                      "element": atoms.get_chemical_symbols()}).head(12))

## Identify molecules
The graph infers connectivity from distances and covalent radii; inspect the component formulas before trusting downstream statistics. A molecular graph is not a bond-order assignment. Incorrect cutoffs, unusual chemistry, or bonds formed during MD need explicit validation.

In [ ]:
from MartiniFuncs.structure import graph
BOND_SCALE = 1.3
bonded = graph.build_bond_graph(atoms, scale=BOND_SCALE)
components = graph.connected_components(bonded, atom_ids=atoms.arrays["global_index"])
lookup = {int(i): row for row, i in enumerate(atoms.arrays["global_index"])}
inventory = pd.DataFrame([
    {"molecule": n, "formula": atoms[[lookup[i] for i in ids]].get_chemical_formula(),
     "atom_count": len(ids), "persistent_ids": ids}
    for n, ids in enumerate(components)
])
display(inventory)

## Stream a short history
Streaming holds one atomic frame at a time; this table grows only with the selected frame count. Full in-memory loading grows with frames × atoms. Splitting into frame files is optional and adds filesystem I/O.

No force field is evaluated. Temperature is shown only when velocities and explicit DOF are available. This inspection does not infer pressure or potential energy from forces.

### Mathematics: whole-frame density and kinetic temperature
For $N$ atoms, cell matrix $H$ (Å), masses $m_i$ (amu) and velocities $\mathbf v_i$:
$$V=|\det H|,\qquad \rho=1.66053906660\,\frac{\sum_i m_i}{V}\quad[\mathrm{g/cm^3}],$$
$$K=\tfrac12\sum_i m_i|\mathbf v_i|^2,\qquad T=\frac{2K}{f_{\rm DOF}k_B}.$$
Kinetic energy is evaluated in ASE's internal mass/velocity units, giving eV; $k_B$ is in eV/K. `TEMPERATURE_DOF` supplies $f_{\rm DOF}$; `"3N-3"` sets $f_{\rm DOF}=3N-3$ and `None` omits temperature. **Current notebook calls do not subtract COM velocity.** Choosing $3N-3$ changes only the denominator; it does not remove drift. A drift-corrected calculation instead uses $\mathbf v_i-\mathbf V$, where $\mathbf V=\sum_i m_i\mathbf v_i/\sum_i m_i$, and requires `remove_com=True`. Constraints require their own DOF accounting. Notebook 08 omits this derived temperature and can accept a documented thermo temperature instead.

A displayed whole-run mean is $\bar x=\sum_{j=0}^{T-1}x_j/T$; it averages saved frames, not independent samples.


In [ ]:
from itertools import islice
from MartiniFuncs.statistics import md as stats
rows = []
reader = iter_lammps_frames(files["dump"], units=UNITS, input_datafile=files["data"], type_symbols=TYPE_SYMBOLS)
try:
    for item in islice(reader, MAX_FRAMES):
        row = {"frame": item.index, "time_ps": item.timestep*TIMESTEP_PS,
               "density_g_cm3": stats.get_frame_density(item.atoms)}
        if TEMPERATURE_DOF is not None and "momenta" in item.atoms.arrays:
            row["temperature_K"] = stats.get_frame_temperature(item.atoms, degrees_of_freedom=TEMPERATURE_DOF)
        rows.append(row)
finally:
    reader.close()
display(pd.DataFrame(rows))

**Next:** notebook 03 extracts selected pairs and complete environments. Notebook 04 computes molecular histories over a chosen time window. Never compare forces from one frame to coordinates or atom ordering from another.